***Business task:*** Identify regions where churn is clustering
above the national average signals a network or service quality problem,
not just individual customer dissatisfaction.

In [ ]:
# Cell 1 — Install
!apt-get install -y -qq mysql-server
!pip install -q pymysql cryptography ipython-sql sqlalchemy

In [ ]:
# Downgrade prettytable to a version known to work with ipython-sql
!pip install prettytable==3.10.2

# Alternatively, force output to Pandas to skip the prettytable formatter entirely
%config SqlMagic.autopandas = True

In [ ]:
# Cell 2 — Start MySQL + remove password
!service mysql start
!mysql --defaults-file=/etc/mysql/debian.cnf \
    -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY ''; FLUSH PRIVILEGES;"

In [ ]:
# Cell 3 — Upload file
from google.colab import files
files.upload()

In [ ]:
# Cell 4 — Load database (pure bash, one line)
!mysql -u root < telecom_complete.sql

In [ ]:
# Cell 5 — Connect Magic SQL
%load_ext sql
%sql mysql+pymysql://root@localhost/telecom
%config SqlMagic.displaylimit = 50

In [ ]:
%%sql
SHOW TABLES;

In [ ]:
%%sql
select status
from subscribers
limit 10;

In [ ]:
%%sql
WITH
subscriber_status AS (
  SELECT
    sub.subscriber_id,
    sub.status,
    r.region_id,
    r.region_name,
    r.region_type,
    c.country_name
  FROM   subscribers sub
  JOIN   countries   c ON sub.country_code = c.country_code
  JOIN   regions     r ON c.country_code   = r.country_code
),
regional_stats AS (
  SELECT
    region_id,
    region_name,
    region_type,
    country_name,
    COUNT(*)                                                  AS total_subs,
    COUNT(CASE WHEN status = 'Churned'   THEN 1 END)         AS churned,
    COUNT(CASE WHEN status = 'Suspended' THEN 1 END)         AS suspended,
    COUNT(CASE WHEN status = 'Active'    THEN 1 END)         AS active,
    ROUND(
      100.0 * COUNT(CASE WHEN status = 'Churned' THEN 1 END)
      / NULLIF(COUNT(*), 0)
    , 1)                                                      AS churn_rate_pct
  FROM   subscriber_status
  GROUP  BY region_id, region_name, region_type, country_name
),
national_avg AS (
  SELECT ROUND(AVG(churn_rate_pct), 2) AS avg_churn_pct
  FROM   regional_stats
),
ticket_pressure AS (
  SELECT
    r.region_id,
    COUNT(t.ticket_id)                                        AS open_tickets,
    COUNT(CASE WHEN t.priority IN ('High','Critical')
               THEN 1 END)                                    AS critical_tickets,
    COUNT(CASE WHEN t.category = 'Network'
               THEN 1 END)                                    AS network_tickets
  FROM   support_tickets t
  JOIN   subscribers     sub ON t.subscriber_id = sub.subscriber_id
  JOIN   countries       c   ON sub.country_code = c.country_code
  JOIN   regions         r   ON c.country_code   = r.country_code
  WHERE  t.status IN ('Open','InProgress','Escalated')
  GROUP  BY r.region_id
)
SELECT
  rs.region_name,
  rs.region_type,
  rs.country_name,
  rs.total_subs,
  rs.churned,
  rs.suspended,
  rs.churn_rate_pct,
  na.avg_churn_pct                                            AS national_avg_pct,
  ROUND(rs.churn_rate_pct - na.avg_churn_pct, 1)             AS above_avg_by,
  COALESCE(tp.open_tickets,     0)                           AS open_tickets,
  COALESCE(tp.network_tickets,  0)                           AS network_tickets,
  COALESCE(tp.critical_tickets, 0)                           AS critical_tickets,
  CASE
    WHEN rs.churn_rate_pct > na.avg_churn_pct * 1.5
     AND COALESCE(tp.network_tickets, 0) > 2
      THEN 'Network investigation required'
    WHEN rs.churn_rate_pct > na.avg_churn_pct * 1.5
      THEN 'High churn — service quality review'
    WHEN rs.churn_rate_pct > na.avg_churn_pct
      THEN 'Above average — monitor closely'
    ELSE 'Within normal range'
  END                                                         AS recommended_action,
  DENSE_RANK() OVER (
    ORDER BY rs.churn_rate_pct DESC
  )                                                           AS churn_rank
FROM   regional_stats   rs
CROSS JOIN national_avg na
LEFT JOIN ticket_pressure tp ON rs.region_id = tp.region_id
ORDER  BY rs.churn_rate_pct DESC;